### 0. Dependancies

Installs runtime dependencies used later in the notebook.

Details:
- Installs `protobuf` — a serialization library often required by prebuilt model packages.
- Installs `sentencepiece` — a tokenizer library required by the mBART tokenizer and other Hugging Face tokenizers.


In [6]:
!pip install protobuf
!pip install sentencepiece

### 1. Loading the dataset

### Data representation per subfolder

The SMOL dataset subfolders use slightly different JSON keys. The table below summarizes the expected keys for each folder.

#### GATITOS
| sl | tl | is_source_orig | src | trgs |
|---|---:|---:|---|---|
| en | aa | True | 'how are you' | ['mannah aniih?', 'anninnaay?'] |

#### SmolSent
| id | sl | tl | is_src_orig | src | trg |
|---|---|---:|---:|---|---|
| 381 | en | ber | True | 'Rih, a deaf former soldier, plots rebellion while married to a queer, teenage god.' | 'ⵔⵉⵀ, ⴷ ⴰⴷⴻⴼⵔⵉⵔ ⴰⵇⴱⵓⵔ ⴰⴻⵎⴻⵙⵍⵉ, ⵢⴻⵜⵜⵀⴻⴳⴳⵉ ⵜⴰⴴⴻⵡⵡⴰⵜ-ⵉⵙ, ⴴⴰⵙ ⴰⴽⴽⴻⵏ ⵢⴻⵣⵡⴻⴵ ⴷ ⵢⵉⵡⴻⵏ ⵢⵉⵍⵓ ⵉⵍⴻⵎⵥⵉ ⵉⵁⴻⵎⵎⵍⴻⵏ ⴰⵔⵔⴰⵛ.' |

#### SmolDoc
| id | sl | tl | is_src_orig | factuality | srcs | trgs |
|---|---|---:|---:|---|---|---|
| 'topic_587__weyiwiniwaaotiwenwy' | en | pcm | True | 'ok' | ['"What the hell are you doing, you idiot?!"','"Excuse me?"',...,'"Yeah, I heard of you."'], | ['"Wetin di hell dey do, yu idiot?!"', '"Ekskuse mi?"',..., '"Na so, I don hear yu."'] |

#### Missing values
As is visible in the tables above, the values for script type or language origin are missing. Therefore these values will be added when processing the dataset. 

This code cell handles full dataset ingestion and initial formatting for model training/evaluation.

Detailed steps performed by the cell:
1. Imports standard libraries: `pandas`, `os`, `json`, and `pathlib.Path`.
2. Defines `smol_path = Path('smol')` and sets `subfolders = ['gatitos', 'smoldoc', 'smolsent']`.
3. Implements `load_smol_data()` which:
   - Iterates over each `.jsonl` file in the subfolders.
   - Reads each line as JSON and normalizes keys for source (`src` or `srcs`) and target (`trg` or `trgs`).
   - Appends dictionaries with `source`, `target`, and `category` to a list.
   - Returns a combined `pandas.DataFrame`.
4. Loads the combined DataFrame `smol_df` and converts entries into `formatted_df` suitable for mBART by creating a `translation` list of `{source, target}` dictionaries.
5. Prints the total number of samples, per-category counts, and shows the first few rows.



In [24]:
import pandas as pd
import os
import json
from pathlib import Path

# Path to SMOL dataset
smol_path = Path('smol')

# Subfolders in SMOL dataset
subfolders = ['gatitos', 'smoldoc', 'smolsent']

# Function to read and combine all SMOL files
def load_smol_data():
    data = []
    for subfolder in subfolders:
        subfolder_path = smol_path / subfolder
        for file in subfolder_path.glob('*.jsonl'):
            with open(file, 'r', encoding='utf-8') as f:
                for line in f:
                    json_obj = json.loads(line.strip())
                    # choose correct source key per subfolder
                    if subfolder in ('gatitos', 'smolsent'):
                        source = json_obj.get('src', '')
                    else:  # smoldoc
                        source = json_obj.get('srcs', '')

                    # allow for either 'trgs' or 'trg' for target as a fallback
                    target = json_obj.get('trgs', json_obj.get('trg', ''))

                    data.append({
                        'source': source,
                        'target': target,
                        'category': subfolder
                    })
    
    return pd.DataFrame(data)

# Load the data
smol_df = load_smol_data()

# Format for mBART-50
formatted_data = {
    'translation': []
}

for _, row in smol_df.iterrows():
    formatted_data['translation'].append({
        'source': row['source'],
        'target': row['target']
    })

# Convert to DataFrame for easier handling
formatted_df = pd.DataFrame(formatted_data)

# Display first few entries and statistics
print("Total number of samples:", len(formatted_df))
print("\nSamples per category:")
print(smol_df['category'].value_counts())
formatted_df.head()

Total number of samples: 1498527

Samples per category:
category
gatitos     1299637
smolsent     144958
smoldoc       53932
Name: count, dtype: int64


,translation
0,"{'source': 'Qafar', 'target': ['Afar']}"
1,"{'source': 'Anu', 'target': ['I']}"
2,"{'source': 'Yeey', 'target': ['Yes']}"
3,"{'source': 'Meqeh', 'target': ['Yes']}"
4,"{'source': 'Salaam qalaykum', 'target': ['Hell..."


### Preliminary metadata processing and analysis

It may be useful for us to group the language datasets by script types and region to get an idea of what languages to work with. These can give indicators to what languages we should focus on to generate meaningful results. Additionally, we may use glottocodes from the metadata to retrieve language family, something that could be used to teach mBART a language-family embedding.

The metadata describes the properties of the target language (So for instance, en_af will be latin script and Africa region, since Afrikaans is the target language). This means that if we have two instances of a language as the target, we don't need to count them twice.

#### Script Type
Since the metadata only exists in the source -> target direction, a translation like en_ab will have its target script being Cyrillic, but in the case of the English, we don't know what script type it will be from metadata alone (Although in this case we can infer that it is latin). Even though smol has data for both directions, the metadata will not always be enough. For instance, translations into Kashmiri exist in both Arab and Deva script, and more considerations will have to be made regarding these exceptions.

#### Region
We also need to make considerations about region. This is because region may not be an optimal indicator of language similarity. Take Afrikaans, for instance, which is listed as an African language, but it is derived from Dutch and is therefore a Germanic language. It is likely that Afrikaans may have absorbed words or grammatical structures from surrounding African languages, but to a much lesser extent than say, Xhosa and Zulu.

Another consideration of region are its inter-continental differences. It is likely that there is a greater similarity between languages of Oceania (Largely polynesian ethnic groups and languages) than say, Africa. Europe is another example where both Germanic, Romantic, Slavid and Uralic languages exist, further diluting a shared region embedding.

It may therefore be beneficial to restrict languages to, for example, languages that share region and script, to better account for such errors.

#### Glottocode-derived language family
Lastly, each target language is marked with a glottocode, uniquely identifying it among the languages of the world. This can be used to look up features of the language beyond what is in the metadata, such as language family.

This may be useful from a language-similarity perspective, but we would need to use an outside resource to link glottocodes with their respective language family, as it is not a direct feature of the SMOL dataset (But it can be argued that it is indirectly).

*NOTE:* It currently does not seem to be possible to extract the metadata in an easy way, as it only really exists in the readme file. That's why we simply take that file and parse out the table like this, but we're looking for a better solution.

In [42]:
# Group by script type and display counts
import pandas as pd
import requests
import re
from io import StringIO

# Path to SMOL dataset README
readme_path = Path('smol/README.md')

# Retrieve the README text
with open(readme_path, 'r', encoding='utf-8') as f:
    text = f.read()

# Extract the markdown table section containing the per-language info
lines = text.splitlines()
table_lines = []
in_table = False
for line in lines:
    # Start when LP appears (Beginning of the table)
    if re.match(r"^LP\s*\|", line.strip()):
        in_table = True
        table_lines.append(line)
        continue
    if in_table:
        if re.match(r"^---", line.strip()) or line.strip().startswith("|---"):
            table_lines.append(line)
            continue
        if "|" in line:
            table_lines.append(line)
        else:
            break  # end once we hit a non-table line

# Join the lines into one markdown chunk
md_table = "\n".join(table_lines)

# 3. Parse the markdown table using pandas
df = pd.read_csv(StringIO(md_table), sep="|", engine="python")

# Clean up columns: remove extra whitespace and unnamed cols
df = df.dropna(axis=1, how="all")
df.columns = [c.strip() for c in df.columns]
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# Remove any duplicate language entries.
df_unique = df.drop_duplicates(subset=["Target Language Name"], keep="first")

# These are considered unique, but not necessarily unique glottocodes. This is because
# some entries may have the samee glottocodes but translate into different scripts.
print("Total number of unique target languages:", len(df_unique))

 # Unique glottocodes
print("\nTotal number of unique glottocodes:", df_unique["Glottocode"].nunique())

# Group by script type
print("\nTotal number of script types:", df_unique["ISO 15924 Script"].nunique())
print("Languages per script type:")
print(df_unique["ISO 15924 Script"].value_counts().head(10))

# Group by region
print("\nTotal number of continents:", df_unique["Continent"].nunique())
print("Languages by continent:")
print(df_unique["Continent"].value_counts().head(10))


Total number of unique target languages: 226

Total number of unique glottocodes: 217

Total number of script types: 24
Languages per script type:
ISO 15924 Script
Latn             136
Deva              31
Arab              18
Cyrl              16
Beng               3
Tibt               3
Ethi               2
-------------      1
Limb               1
Knda               1
Name: count, dtype: int64

Total number of continents: 6
Languages by continent:
Continent
Asia           100
Africa          71
Europe          29
Americas        18
Oceania          7
-----------      1
Name: count, dtype: int64


C:\Users\mhusu\AppData\Local\Temp\ipykernel_21652\1826467439.py:42: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


### Analysis

As we can see from the metadata, not every language in the dataset is properly represented, but most are. A total of
221 languages exist in SMOL but the metadata shows 217 unique glottocodes, meaning there are enough unique languages to work with.

From these unique languages we find a variety of script and regions, and with the glottocodes we will be able to group them by language family as well.

Beyond simply creating the embeddings to facilitate this, it may be interesting to discuss the statistics and think about what languages could be meaningful to work with.

It will be interesting to compare large script groups with smaller script groups. It may not be beneficial to work with languages with latin script, since they exist all over the globe and come from a variety of different language families. Instead, choosing to focus on languages that belong to small script categories, such as the Ethi script (Ge'ez), where it is likely that the languages are closely related.

Similarly, it may be interesting to see if smaller region groups give better results than larger region groups.

Finally, it may be interesting to use a glottocode-derived language-family embedding on small or big groups of languages, to see if too few or too many related languages cause a decline in performance over baseline.


### 2. Load mBART-50 model

Loads the mBART-50 tokenizer and model from Hugging Face `transformers`.

Details:
- `MBart50Tokenizer` provides the subword tokenizer compatible with mBART-50.
- `MBartForConditionalGeneration` is the PyTorch model class used for many-to-many translation with mBART.


In [2]:
from transformers import MBartForConditionalGeneration, MBart50Tokenizer

# Load mBART-50 tokenizer and model
tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

C:\Users\kysel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
